# Step 06 — Four search algorithms on London road network


In [53]:
import sys
import pickle
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

import routing
from routing.config import ORIGIN, DESTINATION, RESULTS_DIR
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_pipeline import make_routing_graph
from routing.algorithms import run_all_algorithms
from routing.route_view import add_route_overlay

print("Python:", sys.version)
print("Kernel:", sys.executable)
print("Routing package:", Path(routing.__file__).resolve())


Python: 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]
Kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Routing package: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/__init__.py


## 1 — Load OSM and the saved traffic snapshot first


In [54]:
RADIUS_M = 2000

G = load_road_graph(radius_m=RADIUS_M)

snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"

if not snapshot_path.exists():
    raise FileNotFoundError(
        f"{snapshot_path} was not found. Run Notebook 05 and save a traffic snapshot first."
    )

with snapshot_path.open("rb") as f:
    traffic_bundle = pickle.load(f)

snapshot = traffic_bundle["snapshot"]
summary = traffic_bundle.get("summary", {})
plan = traffic_bundle.get("plan", [])

assert isinstance(G, nx.MultiDiGraph)
assert isinstance(snapshot, nx.MultiDiGraph)

if set(G.nodes) != set(snapshot.nodes):
    raise ValueError("Saved snapshot nodes do not match the newly loaded OSM graph.")

if set(G.edges(keys=True)) != set(snapshot.edges(keys=True)):
    raise ValueError("Saved snapshot directed edges do not match the newly loaded OSM graph.")

print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")
print("Snapshot:", snapshot.number_of_nodes(), "nodes,", snapshot.number_of_edges(), "directed edges")
print("Saved usable observations:", summary.get(
    "cumulative_usable_observed_edges",
    summary.get("usable_observed_edges", "not reported")
))
print("Saved at:", traffic_bundle.get("saved_at_utc", "timestamp not recorded"))

print("PASS: The saved traffic snapshot matches the OSM graph")


OSM graph: 2569 nodes, 5850 directed edges
Snapshot: 2569 nodes, 5850 directed edges
Saved usable observations: 6
Saved at: timestamp not recorded
PASS: The saved traffic snapshot matches the OSM graph


## 2 — Inspect the traffic observations before routing


In [55]:
from collections import Counter

status_counts = Counter()
observed_rows = []

for u, v, key, attrs in snapshot.edges(keys=True, data=True):
    status = attrs.get("traffic_status", "missing")
    status_counts[status] += 1

    usable = (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and status in {"red", "yellow", "clear", "closed"}
    )

    if usable:
        observed_rows.append({
            "edge_id": (u, v, key),
            "status": status,
            "match_source": attrs.get("traffic_match_source"),
            "current_speed_kph": attrs.get("current_speed_kph"),
            "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
            "travel_time_s": attrs.get("travel_time_s"),
            "is_closed": attrs.get("is_closed", False),
        })

print("Traffic status counts:", dict(status_counts))
print("Usable observed directed edges:", len(observed_rows))
print("Total directed edges:", snapshot.number_of_edges())

observed_edges_df = pd.DataFrame(observed_rows)
display(observed_edges_df)


Traffic status counts: {'missing': 5844, 'clear': 4, 'yellow': 2}
Usable observed directed edges: 6
Total directed edges: 5850


,edge_id,status,match_source,current_speed_kph,free_flow_speed_kph,travel_time_s,is_closed
0,"(109869, 26389013, 0)",clear,direct_probe,27.0,27.0,11.756003,False
1,"(20947178, 1691189798, 0)",yellow,reverse_inferred,18.0,26.0,17.870665,False
2,"(385312258, 20947178, 0)",yellow,reverse_inferred,18.0,26.0,18.690325,False
3,"(1356901661, 109869, 0)",clear,direct_probe,27.0,27.0,5.577524,False
4,"(1960925507, 6280192965, 0)",clear,direct_probe,24.0,24.0,18.804389,False
5,"(6280192965, 5898995362, 0)",clear,direct_probe,24.0,24.0,16.052239,False


## 3 — Choose the origin/destination on the map


In [56]:
route_widget, route_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
    max_snap_distance_m=350.0,
)

display(route_widget)


In [57]:
source = route_state["origin_node"]
goal = route_state["destination_node"]

print("Origin node:", source)
print("Destination node:", goal)
print("Last double-click:", route_state["last_click_latlon"])

assert source in G
assert goal in G

print("PASS: Valid origin and destination nodes")


Origin node: 25291778
Destination node: 26389375
Last double-click: None
PASS: Valid origin and destination nodes


## 4 — Build all routing graphs from the loaded traffic snapshot


In [58]:
measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

print("Measured-time edges:", measured_time.number_of_edges())
print("Measured-distance edges:", measured_distance.number_of_edges())
print("Free-flow edges:", free_flow_time.number_of_edges())

print("PASS: Routing graphs were built from the saved traffic snapshot")


Measured-time edges: 5850
Measured-distance edges: 5850
Free-flow edges: 5850
PASS: Routing graphs were built from the saved traffic snapshot


## 5 — Run BFS, DFS, Greedy Best-First Search, and A*


In [59]:
results_time = run_all_algorithms(measured_time, source, goal)
results_distance = run_all_algorithms(measured_distance, source, goal)
results_free = run_all_algorithms(free_flow_time, source, goal)

result_columns = [
    "algorithm",
    "found",
    "hops",
    "distance_m",
    "travel_time_s",
    "objective_cost",
    "imputed_route_edges",
    "expanded_nodes",
    "max_frontier_size",
    "runtime_ms",
]

def results_to_frame(results, scenario_name):
    rows = []
    for name, result in results.items():
        row = {key: result.get(key) for key in result_columns}
        row["scenario"] = scenario_name
        rows.append(row)
    return pd.DataFrame(rows)

comparison_df = pd.concat(
    [
        results_to_frame(results_time, "Measured Time"),
        results_to_frame(results_distance, "Measured Distance"),
        results_to_frame(results_free, "Free Flow"),
    ],
    ignore_index=True,
)

display(comparison_df.round(3))

if results_time["A*"]["found"]:
    print("A* time-optimal path:", results_time["A*"]["path_edges"])
else:
    print("No directed path exists for the selected origin/destination.")

print("PASS: All four algorithms were executed on all three routing graphs")


,algorithm,found,hops,distance_m,travel_time_s,objective_cost,imputed_route_edges,expanded_nodes,max_frontier_size,runtime_ms,scenario
0,bfs,True,5,679.322,81.519,81.519,5,44,18,0.142,Measured Time
1,DFS,True,209,15561.308,1867.357,1867.357,209,610,162,1.793,Measured Time
2,Greedy,True,5,679.322,81.519,81.519,5,6,8,3.969,Measured Time
3,A*,True,6,628.181,75.382,75.382,6,9,10,3.588,Measured Time
4,bfs,True,5,679.322,81.519,679.322,5,44,18,0.139,Measured Distance
5,DFS,True,209,15561.308,1867.357,15561.308,209,610,162,1.951,Measured Distance
6,Greedy,True,5,679.322,81.519,679.322,5,6,8,4.557,Measured Distance
7,A*,True,6,628.181,75.382,628.181,6,9,10,3.740,Measured Distance
8,bfs,True,5,679.322,81.519,81.519,5,44,18,0.127,Free Flow
9,DFS,True,209,15561.308,1867.357,1867.357,209,610,162,1.943,Free Flow


A* time-optimal path: [(25291778, 1360237602, 0), (1360237602, 20963439, 0), (20963439, 6280192965, 0), (6280192965, 6280192964, 0), (6280192964, 34519894, 0), (34519894, 26389375, 0)]
PASS: All four algorithms were executed on all three routing graphs


## 6 — Draw the time-objective routes on the map


In [60]:
for name, result in results_time.items():
    if result["found"]:
        add_route_overlay(
            route_widget,
            G,
            result,
            name=f"{name} | measured-time weights",
        )
        print(name, "route drawn; edges:", len(result["path_edges"]))
    else:
        print(name, "did not find a route")

print("Routes were added to the interactive map.")


BFS route drawn; edges: 5
DFS route drawn; edges: 209
Greedy route drawn; edges: 5
A* route drawn; edges: 6
Routes were added to the interactive map.


## 7 — Simulate one directed road closure


In [61]:
best_route = results_time["A*"]

if best_route["found"] and best_route["path_edges"]:
    chosen_closure = best_route["path_edges"][len(best_route["path_edges"]) // 2]

    closure_graph = make_routing_graph(
        snapshot,
        objective="time",
        scenario="closure",
        imposed_closure=chosen_closure,
    )

    assert not closure_graph.has_edge(*chosen_closure)

    closure_results = run_all_algorithms(closure_graph, source, goal)

    print("Simulated directed closure:", chosen_closure)
    display(pd.DataFrame([
        {
            "algorithm": name,
            "found": result["found"],
            "hops": result["hops"],
            "travel_time_s": result["travel_time_s"],
            "distance_m": result["distance_m"],
        }
        for name, result in closure_results.items()
    ]).round(3))
else:
    print("Closure simulation skipped because A* did not find a nonempty route.")


Simulated directed closure: (6280192965, 6280192964, 0)


,algorithm,found,hops,travel_time_s,distance_m
0,BFS,True,5,81.519,679.322
1,DFS,True,209,1867.357,15561.308
2,Greedy,True,5,81.519,679.322
3,A*,True,5,81.519,679.322


## 8 — Validate route structures


In [62]:
for graph, results in (
    (measured_time, results_time),
    (measured_distance, results_distance),
    (free_flow_time, results_free),
):
    for name, result in results.items():
        if not result["found"]:
            continue

        assert result["path_nodes"][0] == source
        assert result["path_nodes"][-1] == goal
        assert len(result["path_edges"]) == len(result["path_nodes"]) - 1
        assert all(graph.has_edge(*edge) for edge in result["path_edges"])
        assert all(G.has_edge(*edge) for edge in result["path_edges"])

print("PASS: All found routes have valid directed edge sequences")


PASS: All found routes have valid directed edge sequences


## 9 — Build an Observed-Only graph




In [63]:
def build_observed_only_graph(source_graph):
    graph = source_graph.copy()
    allowed_sources = {"direct_probe", "reverse_inferred"}
    usable_statuses = {"red", "yellow", "clear"}

    remove_edges = []

    for u, v, key, attrs in graph.edges(keys=True, data=True):
        valid_observation = (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status") in usable_statuses
            and attrs.get("traffic_match_source") in allowed_sources
            and attrs.get("is_closed") is not True
        )

        if not valid_observation:
            remove_edges.append((u, v, key))

    graph.remove_edges_from(remove_edges)
    return graph


observed_only = build_observed_only_graph(snapshot)

print("========== OBSERVED-ONLY GRAPH ==========")
print("Nodes:", observed_only.number_of_nodes())
print("Edges:", observed_only.number_of_edges())

observed_route_exists = (
    source in observed_only
    and goal in observed_only
    and nx.has_path(observed_only, source, goal)
)

print("Observed-only route exists:", observed_route_exists)

if not observed_route_exists:
    print(
        "Expected with sparse coverage: observed edges exist, "
        "but they do not yet form a continuous directed route."
    )


========== OBSERVED-ONLY GRAPH ==========
Nodes: 2569
Edges: 6
Observed-only route exists: False
Expected with sparse coverage: observed edges exist, but they do not yet form a continuous directed route.


## 10 — Route Audit: inspect the exact edges used by each algorithm


In [64]:
route_audit_rows = []

scenario_results = {
    "Measured Time": results_time,
    "Measured Distance": results_distance,
    "Free Flow": results_free,
}

for scenario_name, results in scenario_results.items():
    for algorithm, result in results.items():
        if not result["found"]:
            continue

        for u, v, key in result["path_edges"]:
            attrs = snapshot[u][v][key]

            usable_observation = (
                attrs.get("traffic_matched") is True
                and attrs.get("time_is_imputed") is False
                and attrs.get("traffic_status") in {"red", "yellow", "clear"}
                and attrs.get("traffic_match_source") in {
                    "direct_probe",
                    "reverse_inferred",
                }
            )

            route_audit_rows.append({
                "scenario": scenario_name,
                "algorithm": algorithm,
                "edge_id": (u, v, key),
                "traffic_status": attrs.get("traffic_status", "missing"),
                "match_source": attrs.get("traffic_match_source"),
                "match_reason": attrs.get("traffic_match_reason"),
                "current_speed_kph": attrs.get("current_speed_kph"),
                "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
                "travel_time_s": attrs.get("travel_time_s"),
                "imputed": attrs.get("time_is_imputed", True),
                "usable_observation": usable_observation,
            })

route_audit_df = pd.DataFrame(route_audit_rows)

print("Route-audit rows:", len(route_audit_df))
display(route_audit_df)


Route-audit rows: 675


,scenario,algorithm,edge_id,traffic_status,match_source,match_reason,current_speed_kph,free_flow_speed_kph,travel_time_s,imputed,usable_observation
0,Measured Time,BFS,"(25291778, 5898995362, 0)",missing,None,None,None,None,13.720352,True,False
1,Measured Time,BFS,"(5898995362, 2360746811, 0)",missing,None,None,None,None,1.153502,True,False
2,Measured Time,BFS,"(2360746811, 6280192964, 0)",missing,None,None,None,None,12.858625,True,False
3,Measured Time,BFS,"(6280192964, 34519894, 0)",missing,None,None,None,None,32.019111,True,False
4,Measured Time,BFS,"(34519894, 26389375, 0)",missing,None,None,None,None,21.767099,True,False
...,...,...,...,...,...,...,...,...,...,...,...
670,Free Flow,A*,"(1360237602, 20963439, 0)",missing,None,None,None,None,5.699065,True,False
671,Free Flow,A*,"(20963439, 6280192965, 0)",missing,None,None,None,None,6.637069,True,False
672,Free Flow,A*,"(6280192965, 6280192964, 0)",missing,None,None,None,None,1.171335,True,False
673,Free Flow,A*,"(6280192964, 34519894, 0)",missing,None,None,None,None,32.019111,True,False


In [65]:
if not route_audit_df.empty:
    route_coverage_df = (
        route_audit_df
        .groupby(["scenario", "algorithm"], as_index=False)
        .agg(
            route_edges=("edge_id", "count"),
            observed_edges=("usable_observation", "sum"),
            imputed_edges=("imputed", "sum"),
        )
    )

    route_coverage_df["observed_coverage_percent"] = (
        100 * route_coverage_df["observed_edges"]
        / route_coverage_df["route_edges"]
    ).round(2)

    print("========== ROUTE TRAFFIC COVERAGE ==========")
    display(route_coverage_df)
else:
    print("No route edges were available for audit.")


========== ROUTE TRAFFIC COVERAGE ==========


,scenario,algorithm,route_edges,observed_edges,imputed_edges,observed_coverage_percent
0,Free Flow,A*,6,0,6,0.0
1,Free Flow,BFS,5,0,5,0.0
2,Free Flow,DFS,209,0,209,0.0
3,Free Flow,Greedy,5,0,5,0.0
4,Measured Distance,A*,6,0,6,0.0
5,Measured Distance,BFS,5,0,5,0.0
6,Measured Distance,DFS,209,0,209,0.0
7,Measured Distance,Greedy,5,0,5,0.0
8,Measured Time,A*,6,0,6,0.0
9,Measured Time,BFS,5,0,5,0.0


## 11 — Prepare route-focused probes for the next collection round



In [66]:
from routing.map_matching import directed_osm_line

route_probe_rows = []

astar_result = results_time["A*"]

if astar_result["found"]:
    for edge_id in astar_result["path_edges"]:
        u, v, key = edge_id
        attrs = snapshot[u][v][key]

        usable_observation = (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status") in {"red", "yellow", "clear"}
            and attrs.get("traffic_match_source") in {
                "direct_probe",
                "reverse_inferred",
            }
        )

        if usable_observation:
            continue

        midpoint = directed_osm_line(G, u, v, key).interpolate(
            0.5,
            normalized=True,
        )

        route_probe_rows.append({
            "edge_id": edge_id,
            "latitude": round(midpoint.y, 7),
            "longitude": round(midpoint.x, 7),
            "length_m": round(float(attrs.get("length", 0.0)), 2),
            "road_class": attrs.get("highway"),
            "current_traffic_status": attrs.get("traffic_status", "missing"),
            "time_is_imputed": attrs.get("time_is_imputed", True),
        })

route_probe_candidates_df = pd.DataFrame(route_probe_rows)
route_probe_edge_ids = [
    tuple(row["edge_id"]) for row in route_probe_rows
]

print("A* route edges needing traffic observations:", len(route_probe_edge_ids))
display(route_probe_candidates_df)

print(
    "Next step: use route_probe_edge_ids as selected_edge_ids in Notebook 05 "
    "for a bounded route-focused collection."
)


import json

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"
route_probe_payload = [
    {**row, "edge_id": list(row["edge_id"])}
    for row in route_probe_rows
]
route_probe_path.parent.mkdir(parents=True, exist_ok=True)
route_probe_path.write_text(
    json.dumps(route_probe_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Route-focused probe candidates exported:", route_probe_path)


A* route edges needing traffic observations: 6


,edge_id,latitude,longitude,length_m,road_class,current_traffic_status,time_is_imputed
0,"(25291778, 1360237602, 0)",51.494826,-0.174412,67.40,primary,missing,True
1,"(1360237602, 20963439, 0)",51.495106,-0.173747,47.49,primary,missing,True
2,"(20963439, 6280192965, 0)",51.495457,-0.173490,55.31,living_street,missing,True
3,"(6280192965, 6280192964, 0)",51.495749,-0.173533,9.76,living_street,missing,True
4,"(6280192964, 34519894, 0)",51.496979,-0.173807,266.83,"[unclassified, living_street]",missing,True
5,"(34519894, 26389375, 0)",51.498980,-0.174206,181.39,unclassified,missing,True


Next step: use route_probe_edge_ids as selected_edge_ids in Notebook 05 for a bounded route-focused collection.
Route-focused probe candidates exported: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/route_probe_candidates.json
